In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import os

import tiktoken

In [2]:
train_data = pd.read_csv("../data/train.csv")
train_data.head()

,paper_id,text,summary
0,0,## FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCI...,"In this article, Victor Fan argues that analys..."
1,1,## 1. Introduction\n\n\nAn Electronic Health R...,Problem definition: Physicians spend more than...
2,2,## Introduction\n\n\nTranslation plays an i...,Literary translation is one of the most challe...
3,3,## 1 Problem Setup\n\n\nRecent political scien...,There is a long-running debate on evaluating f...
4,4,## INTRODUCTION\n\n\nThis article investigat...,"Recently, ‘bimajyo’ (美魔女) came into focus in J..."


In [3]:
df = train_data.copy()

In [4]:
df['text_words'] = df['text'].apply(lambda x: len(str(x).split()))
df['summary_words'] = df['summary'].str.split().apply(len)
df['compression_ratio'] = df['summary_words'] / df['text_words']

In [5]:
enc = tiktoken.get_encoding("cl100k_base")
df['text_tokens'] = df['text'].apply(lambda x: len(enc.encode(x)))

In [6]:
# Function to count tokens in a text
def count_tokens(text):
    enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))

# Chunking function to split text into smaller parts based on token count
def chunk_text(text, max_tokens = 2500):
    paragraphs = [p for p in text.split("\n\n") if p.strip()]
    chunks = []
    current_chunk = []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = count_tokens(para)

        # Fallback: if a single paragraph exceeds max_tokens, we will hard split it
        if para_tokens > max_tokens:
            if current_chunk:
                chunks.append("\n\n".join(current_chunk))
                current_chunk = []
                current_tokens = 0

            # Split the paragraph into smaller parts
            words = para.split()
            sub_chunk = []
            sub_tokens = 0
            for w in words:
                w_tokens = count_tokens(w)

                # If adding this word exceeds the max_tokens, finalize the current sub_chunk
                if sub_tokens + w_tokens > max_tokens:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk = []
                    sub_tokens = 0

                sub_chunk.append(w)
                sub_tokens += w_tokens

            if sub_chunk:
                chunks.append(" ".join(sub_chunk))
            continue

        #Normal Case: does this paragraph fit in the current chunk?
        if current_tokens + para_tokens > max_tokens:
            # If not, finalize the current chunk and start a new one
            chunks.append("\n\n".join(current_chunk))
            current_chunk = [para]
            current_tokens = para_tokens
        else:
            # If it fits, add it to the current chunk
            current_chunk.append(para)
            current_tokens += para_tokens

    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return chunks


In [7]:
MAP_PROMPT_TEMPLATE = """
You are summarizing one section of a longer academic paper. This is section {chunk_num} of {total_chunks}.

Write a concise summary of ONLY the content in this section. Do not add information not present in the text. Do not refer to "this section" or "this chunk" — write as if summarizing standalone content.

Section text:
{chunk_text}

Summary:"""

## For Groq

In [ ]:
from groq import Groq

from dotenv import load_dotenv
load_dotenv()

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

response = client.chat.completions.create(
    messages=[{"role": "user", "content": "Say hello"}],
    model="openai/gpt-oss-20b",
)
print(response.choices[0].message.content)

Hello! How can I help you today?


## For Cerebras

In [8]:

from cerebras.cloud.sdk import Cerebras

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Why is fast inference important?",
        }
],
    model="gpt-oss-120b",
)

print(chat_completion.choices[0].message.content)

## Quick‑Answer Summary
Fast inference – the ability of a model to produce predictions with minimal latency and computational cost – matters because **real‑world systems are judged by what they can do, not just by how accurately they can learn**.  In production, every millisecond, watt, or dollar saved can translate into:

| Domain | Why Speed Matters | Concrete Impact |
|--------|-------------------|-----------------|
| **User‑facing services** (search, recommendation, chat, vision) | Low latency → better user experience, higher conversion, lower churn | A 100 ms delay in a product recommendation can cut revenue by ~5 % (Amazon, 2022). |
| **Real‑time control** (autonomous driving, robotics, finance) | Decisions must be made within strict deadlines to stay safe or profitable | A self‑driving car must react ≤ 50 ms to avoid a collision; latency beyond that is unsafe. |
| **Edge & IoT devices** (phones, wearables, sensors) | Limited compute, memory, bandwidth, and battery | A speech‑to‑

In [9]:
def summarize_chunk(chunk_text, chunk_num, total_chunks):
    prompt = MAP_PROMPT_TEMPLATE.format(
        chunk_num=chunk_num,
        total_chunks=total_chunks,
        chunk_text=chunk_text
    )

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="gpt-oss-120b",
        temperature=0.3,
        max_tokens=1500,
        reasoning_effort="low"
    )
    if chat_completion.choices[0].finish_reason == "length":
        print(f"WARNING: truncated output for this call")

    return chat_completion.choices[0].message.content

In [10]:
chunks = chunk_text(df.loc[0, 'text'])
chunk_summaries = [summarize_chunk(c, i+1, len(chunks)) for i, c in enumerate(chunks)]
# Add a check right after the map step, before reduce:
for i, s in enumerate(chunk_summaries):
    if not s or not s.strip():
        print(f"WARNING: empty summary for chunk {i+1}")

In [11]:
# # Rerun summarize_chunk on that specific chunk in isolation, with full visibility
# response = client.chat.completions.create(
#     messages=[{"role": "user", "content": MAP_PROMPT_TEMPLATE.format(
#         chunk_num=3, total_chunks=len(chunks), chunk_text=chunks[2]
#     )}],
#     model="openai/gpt-oss-20b",
#     temperature=0.3,
#     max_tokens=800,
#     reasoning_effort="low",
# )

# print("finish_reason:", response.choices[0].finish_reason)
# print("content:", repr(response.choices[0].message.content))

In [12]:
REDUCE_PROMPT_TEMPLATE = """You are writing the final summary of an academic paper, based on summaries of its individual sections below.

Combine the section summaries into a single, coherent, densely-written abstract of the paper. Remove redundancy across sections. Do not simply concatenate the section summaries — synthesize them into a unified narrative that reads as if written by the paper's author. Do not add information not present in the section summaries.

Here is an example of the input format and the expected output style:

Example section summaries:
{example_chunk_summaries}

Example final summary:
{example_reference_summary}

Now do the same for the following paper.

Section summaries:
{combined_summaries}

Final summary:"""

In [13]:
def reduce_summaries(chunk_summaries: list[str], 
                     example_chunk_summaries: str, 
                     example_reference_summary: str) -> str:
    
    combined = "\n\n".join(
        f"Section {i+1}: {s}" for i, s in enumerate(chunk_summaries)
    )
    prompt = REDUCE_PROMPT_TEMPLATE.format(
        example_chunk_summaries=example_chunk_summaries,
        example_reference_summary=example_reference_summary,
        combined_summaries=combined,
    )

    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="gpt-oss-120b",
        temperature=0.3,
        max_tokens=1500,
    )

    if response.choices[0].finish_reason == "length":
        print(f"WARNING: truncated output for this call")
        
    return response.choices[0].message.content

In [14]:
import pandas as pd
from tqdm import tqdm
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge2'], use_stemmer=True)

def run_pipeline(text: str, example_chunk_summaries: str, example_reference_summary: str) -> str:
    chunks = chunk_text(text)
    chunk_summaries = [
        summarize_chunk(c, i+1, len(chunks)) for i, c in enumerate(chunks)
    ]
    return reduce_summaries(chunk_summaries, example_chunk_summaries, example_reference_summary)

def evaluate_row(row, example_chunk_summaries, example_reference_summary) -> dict:

    generated = run_pipeline(row['text'], example_chunk_summaries, example_reference_summary)
    scores = scorer.score(row['summary'], generated)
    return {
        'paper_id': row['paper_id'],
        'text_words': row['text_words'],
        'compression_ratio': row['compression_ratio'],
        'generated': generated,
        'generated_words': len(generated.split()),
        'reference_words': row['summary_words'],
        'rouge2_precision': scores['rouge2'].precision,
        'rouge2_recall': scores['rouge2'].recall,
        'rouge2_fmeasure': scores['rouge2'].fmeasure,
    }

In [15]:
df['cr_decile'] = pd.qcut(df['compression_ratio'], 10, labels=False)
sample = df.groupby('cr_decile', group_keys=False).apply(lambda g: g.sample(1, random_state=42))

### Median summary words

In [16]:
example_idx = df.iloc[(df['text_words'] - df['text_words'].median()).abs().argsort()[:5]].index
# then eyeball a few candidates and pick one with a clean, well-formed reference summary
example_idx

Index([337, 756, 713, 216, 805], dtype='int64')

In [17]:
example_idx = 713
example_reference_summary = df.loc[example_idx, 'summary']
example_chunks = chunk_text(df.loc[example_idx, 'text'])

example_summaries = [
    summarize_chunk(c, i+1, len(example_chunks)) for i, c in enumerate(example_chunks)
]

example_chunk_summaries = "\n\n".join(
    f"Section {i+1}: {s}" for i, s in enumerate(example_summaries)
)

In [18]:
results = [evaluate_row(row, example_chunk_summaries, example_reference_summary) for _, row in tqdm(sample.iterrows(), total=len(sample))]
results_df = pd.DataFrame(results)
results_df[['generated_words','reference_words', 'rouge2_precision', 'rouge2_recall', 'rouge2_fmeasure']].describe()

 70%|███████   | 7/10 [08:01<03:26, 68.77s/it]


RateLimitError: Error code: 429 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}

In [20]:
results_df[['text_words', 'compression_ratio', 'rouge2_fmeasure']].corr()

,text_words,compression_ratio,rouge2_fmeasure
text_words,1.000000,-0.606952,0.284188
compression_ratio,-0.606952,1.000000,0.354218
rouge2_fmeasure,0.284188,0.354218,1.000000


In [21]:
results_df['length_ratio'] = results_df['generated_words'] / results_df['reference_words']
print(results_df['length_ratio'].describe())

count    10.000000
mean      1.388685
std       0.436739
min       0.833887
25%       1.240357
50%       1.291333
75%       1.427563
max       2.521739
Name: length_ratio, dtype: float64


In [27]:
results_df.iloc[5]['generated']

'I’m ready to synthesize a concise, densely‑written abstract for your paper, but the material you provided is incomplete.\u202fSection\u202f2 (and any additional sections that may follow) is missing, and without that content I can’t accurately combine all of the section summaries into a single, coherent narrative.\n\nCould you please supply the missing section summary (or any other sections you’d like included)? Once I have the full set of summaries, I’ll be able to produce the unified abstract you’re looking for.'

### Minimum summary words

In [20]:
example_idx_min = df.loc[df['summary_words'] == min(df['summary_words'])].index[0]

example_reference_summary_min = df.loc[example_idx_min, 'summary']
example_chunks_min = chunk_text(df.loc[example_idx_min, 'text'])
example_summaries_min = [
    summarize_chunk(c, i+1, len(example_chunks_min)) for i, c in enumerate(example_chunks_min)
]

example_chunk_summaries_min = "\n\n".join(
    f"Section {i+1}: {s}" for i, s in enumerate(example_summaries_min)
)

In [21]:
results_df_min = [evaluate_row(row, example_chunk_summaries_min, example_reference_summary_min) for _, row in tqdm(sample.iterrows(), total=len(sample))]
results_df_min = pd.DataFrame(results_df_min)
results_df_min[['generated_words','reference_words', 'rouge2_precision', 'rouge2_recall', 'rouge2_fmeasure']].describe()

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [09:05<00:00, 54.59s/it]


,generated_words,reference_words,rouge2_precision,rouge2_recall,rouge2_fmeasure
count,10.00000,10.000000,10.000000,10.000000,10.000000
mean,213.90000,173.500000,0.063956,0.076855,0.058846
std,83.08289,66.431669,0.033623,0.036358,0.020182
min,28.00000,70.000000,0.029851,0.013953,0.024793
25%,222.25000,147.750000,0.037024,0.056799,0.051176
50%,243.50000,158.000000,0.050239,0.074953,0.056685
75%,257.50000,203.500000,0.088398,0.090250,0.070558
max,289.00000,301.000000,0.120370,0.136986,0.093137


In [22]:
results_df_min[['text_words', 'compression_ratio', 'rouge2_fmeasure']].corr()

,text_words,compression_ratio,rouge2_fmeasure
text_words,1.000000,-0.606952,-0.141706
compression_ratio,-0.606952,1.000000,0.303060
rouge2_fmeasure,-0.141706,0.303060,1.000000


In [23]:
results_df_min['length_ratio'] = results_df_min['generated_words'] / results_df_min['reference_words']
print(results_df_min['length_ratio'].describe())

count    10.000000
mean      1.542159
std       1.040585
min       0.130841
25%       0.903931
50%       1.513938
75%       1.819728
max       3.714286
Name: length_ratio, dtype: float64


In [28]:
for i, row in results_df_min.sort_values('length_ratio').iterrows():
    print(i, row['paper_id'], row['length_ratio'])

# then print full generated text for the extremes

6 768 0.1308411214953271
7 842 0.404
9 848 0.760797342192691
8 831 1.3333333333333333
4 855 1.4186046511627908
3 834 1.609271523178808
0 790 1.6666666666666667
1 835 1.870748299319728
2 820 2.5130434782608697
5 867 3.7142857142857144


In [29]:
results_df_min.iloc[6]['generated']

'This study investigates Turkish university EFL learners’ beliefs about translation courses and how those expectations can be aligned with program requirements. Drawing on the Holistic Model (PACTE, 199'

In [30]:
results_df_min.iloc[5]['generated']

'Colombia’s sizable informal economy and entrenched socioeconomic inequities disproportionately affect domestic workers, a predominantly female cohort (≈\u202f677\u202f000 workers, 96\u202f% women) that accounts for 3\u202f% of national employment yet largely lacks formal contracts, social‑security coverage, and wages above the legal minimum. Although the 1951 Labor Code and Law\u202f100 (1993) theoretically extend basic employment rights, maternity/paternity protections, and universal health coverage to formal workers, domestic workers are excluded from compulsory rest‑day, injury, and unemployment benefits. Recent advances—such as the 2015‑onward expansion of early‑childhood and maternal health services, the 18‑week paid maternity leave (Law\u202f1822), and the Domestic Workers Bonus Law (2016)—signal policy intent, while Colombia’s ratification of ILO Convention\u202f189 (2014) and the establishment of the Domestic Workers Bonus underscore formal recognition. Nonetheless, the Global